[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap10_lora_unsloth.ipynb)

# Capítulo 10 — Treino LoRA com Unsloth

> **Baseado no projeto [AI-Orchestrator](https://github.com/aejepsen/AI-Orchestrator) de Anderson Ejepsen**

Notebook completo de fine-tuning LoRA usando Unsloth. Adaptado do notebook de produção `colab_train_lora.ipynb` do AI-Orchestrator, com explicações detalhadas de cada decisão.

**Runtime recomendado: A100 40GB** (LoRA bf16 em 9B consome ~31.8 GB de pico).

### Pré-requisitos
- Dataset SFT no Google Drive: `MyDrive/ai-orchestrator-dataset/orch_sft_train.jsonl` e `orch_sft_val.jsonl`
- Se não tiver o dataset, o notebook cria dados de exemplo para demonstração

## 1. Instalação pinada

Qwen3.5 exige `transformers>=5.2.0` (doc oficial Unsloth). Após rodar esta célula, faça **Ambiente de execução -> Reiniciar sessão** (NÃO "Desconectar e excluir"), depois rode a célula 1b.

**Por que pinar versões?** O ecossistema Unsloth + transformers + trl evolui rápido e combinações incompatíveis quebram silenciosamente. Pinar garante reprodutibilidade.

In [ ]:
# Instalação pinada — Qwen3.5 exige transformers v5
# APÓS RODAR: Ambiente de execução -> Reiniciar sessão, depois rode célula 1b
!pip uninstall -y -q torchaudio 2>/dev/null
!pip install -q --no-cache-dir "unsloth" "unsloth_zoo"
!pip install -q "transformers>=5.2.0,<=5.5.0" "trl>=0.18.2,<=0.24.0" "datasets>=3.4.1,<4.4.0" "accelerate>=1.2" "peft>=0.16" sentencepiece "protobuf<7"
print("INSTALADO — agora: Ambiente de execução -> Reiniciar sessão, depois rode célula 1b")

## 1b. Verificação pós-restart

Rode esta célula **após reiniciar a sessão**. Importar `unsloth` antes de `transformers` é obrigatório (Unsloth patcha o HuggingFace na importação).

In [ ]:
# Verificação pós-restart — rode APÓS reiniciar a sessão
import unsloth  # PRIMEIRO, antes de transformers (obrigatório)
import transformers, trl, datasets as ds_lib
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('datasets:', ds_lib.__version__)
assert transformers.__version__.split('.')[0] == '5', 'transformers errado — reinstale'
print('OK — ambiente pronto')

## 2. Montar Drive e carregar dataset

O dataset é carregado manualmente com `json.loads` (não `datasets.load_dataset`), porque o Arrow não consegue inferir o schema dos `tool_calls` heterogêneos — essa é uma gotcha real que custou horas de debug no projeto.

Se o dataset do Drive não existir, criamos dados de exemplo para demonstração.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os

DATA_DIR = '/content/drive/MyDrive/ai-orchestrator-dataset'

def load_rows(path):
    """Carregamento manual JSONL — Arrow falha em tool_calls heterogêneos."""
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

# Tentar carregar dataset real do Drive
train_path = f'{DATA_DIR}/orch_sft_train.jsonl'
val_path = f'{DATA_DIR}/orch_sft_val.jsonl'

if os.path.exists(train_path) and os.path.exists(val_path):
    raw = {
        'train': load_rows(train_path),
        'val':   load_rows(val_path),
    }
    print(f"Dataset real carregado do Drive")
else:
    # Dataset de demonstração (para quem não tem o dataset do AI-Orchestrator)
    print("Dataset do AI-Orchestrator não encontrado no Drive.")
    print("Criando dataset de demonstração...")
    demo_examples = [
        {"messages": [
            {"role": "system", "content": "Voc\u00ea \u00e9 um assistente corporativo."},
            {"role": "user", "content": "Qual o saldo do cliente Jo\u00e3o?"},
            {"role": "assistant", "content": "O saldo do cliente Jo\u00e3o \u00e9 R$ 15.432,00."}
        ]}
    ] * 100  # repetição para ter volume mínimo
    raw = {
        'train': demo_examples[:90],
        'val':   demo_examples[90:],
    }

print({k: len(v) for k, v in raw.items()})
print('Exemplo (roles):', [m['role'] for m in raw['train'][0]['messages']])

## 3. Carregar modelo base + configurar LoRA

### Decisões de arquitetura (do PLANO_LORA_9B.md):

- **`load_in_4bit=False`**: QLoRA 4-bit é contraindicado pela Unsloth em Qwen3.5 (arquitetura híbrida DeltaNet)
- **`load_in_16bit=True`**: LoRA bf16 é o recomendado
- **`r=16, alpha=32`**: escala alpha/r = 2.0 (padrão para SFT)
- **target_modules**: atenção (q/k/v/o_proj) + MLP (gate/up/down_proj). Camadas DeltaNet ficam fora
- **gradient_checkpointing="unsloth"**: reduz VRAM trocando por recomputação
- **max_seq_length=4096**: cobre system + tools + respostas sem estourar memória

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 4096  # cobre system + tools + respostas curtas dos microsserviços

# Carregar modelo base
# Para demonstração em T4, troque para "unsloth/Qwen2.5-1.5B"
MODEL_NAME = "unsloth/Qwen3.5-9B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ,
    dtype           = torch.bfloat16,  # bf16 nativo
    load_in_4bit    = False,           # QLoRA contraindicado em Qwen3.5
    load_in_16bit   = True,            # LoRA bf16 recomendado
    full_finetuning = False,           # LoRA, não full FT
)

# Aplicar adapters LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,       # rank das matrizes de adaptação
    lora_alpha     = 32,       # escala: alpha/r = 2.0
    lora_dropout   = 0.05,     # regularização leve
    target_modules = [         # doc oficial Unsloth p/ Qwen3.5
        "q_proj", "k_proj", "v_proj", "o_proj",   # atenção
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # reduz VRAM ~30%
    random_state   = 42,
    max_seq_length = MAX_SEQ,
)

model.print_trainable_parameters()

## 4. Formatação do dataset

Aplicamos o chat template do modelo a cada exemplo. O tokenizer do Qwen usa formato ChatML com marcadores `<|im_start|>` e `<|im_end|>`.

**Gotcha**: se o `content` de uma mensagem não for string (ex: dict em tool result), o `apply_chat_template` falha. Convertemos para JSON antes.

In [ ]:
from datasets import Dataset, DatasetDict
import json

def format_row(row):
    """Aplica chat template, convertendo content não-string para JSON."""
    msgs = []
    for m in row['messages']:
        mc = dict(m)
        if not isinstance(mc.get('content'), str):
            mc['content'] = json.dumps(mc.get('content'), ensure_ascii=False)
        msgs.append(mc)
    return tokenizer.apply_chat_template(msgs, tokenize=False)

dataset = DatasetDict({
    'train': Dataset.from_dict({'text': [format_row(r) for r in raw['train']]}),
    'val':   Dataset.from_dict({'text': [format_row(r) for r in raw['val']]}),
})

print(dataset)
print(f"\nExemplo formatado (primeiros 500 chars):")
print(dataset['train'][0]['text'][:500])

## 5. Configurar SFTTrainer

### Hiperparâmetros (justificativa de cada um):

| Parâmetro | Valor | Razão |
|----------|-------|-------|
| `per_device_train_batch_size` | 2 | Máximo que cabe em A100 40GB com 9B bf16 |
| `gradient_accumulation_steps` | 8 | Batch efetivo = 2*8 = 16 (bom para SFT) |
| `num_train_epochs` | 2 | 2 epochs sem overfit (val loss monitorado) |
| `learning_rate` | 2e-4 | Padrão Unsloth para LoRA |
| `lr_scheduler_type` | cosine | Decay suave, melhor que linear para SFT |
| `warmup_ratio` | 0.03 | ~10 steps de warmup |
| `optim` | adamw_8bit | Reduz VRAM do otimizador sem perda de qualidade |
| `bf16` | True | Precisão nativa do modelo |
| `eval_strategy` | epoch | Avalia val loss a cada epoch (overfit check) |

### train_on_responses_only
Mascara o loss fora dos turnos do assistant. Tool results e system prompt não contribuem para o loss — o modelo só aprende a gerar respostas.

**IMPORTANTE**: `output_dir` deve apontar para o Google Drive. Se apontar para `/content` (disco efêmero), perderá checkpoints ao desconectar. Isso custou 2h17 no projeto real.

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

# IMPORTANTE: checkpoints no Drive, NÃO em /content (disco efêmero)
DRIVE_OUT = '/content/drive/MyDrive/ai-orchestrator-lora/training'
os.makedirs(DRIVE_OUT, exist_ok=True)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset  = dataset['val'],
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_length                  = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,    # batch efetivo 16
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        lr_scheduler_type           = 'cosine',
        warmup_ratio                = 0.03,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        bf16                        = True,
        logging_steps               = 10,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        output_dir                  = DRIVE_OUT,  # Drive, não disco efêmero!
        seed                        = 42,
        report_to                   = 'none',
    ),
)

# Mascarar loss fora dos turnos do assistant (ChatML)
# Tool results viram turno "user" no template, logo também ficam mascarados
try:
    from unsloth.chat_templates import train_on_responses_only
    if '<|im_start|>' in (tokenizer.chat_template or ''):
        trainer = train_on_responses_only(
            trainer,
            instruction_part = '<|im_start|>user\n',
            response_part    = '<|im_start|>assistant\n',
        )
        print('train_on_responses_only: ATIVO')
    else:
        print('AVISO: template sem marcadores ChatML — treinando sequência completa')
except Exception as exc:
    print(f'AVISO: train_on_responses_only indisponível ({exc}) — treinando sequência completa')

print(f'\nCheckpoints serão salvos em: {DRIVE_OUT}')

## 6. Treinar!

O treino do AI-Orchestrator (3.050 exemplos, 2 epochs) levou **148 minutos** no A100 40GB, com VRAM de pico de 31.8 GB.

### O que monitorar durante o treino:
- **train_loss**: deve cair consistentemente
- **eval_loss**: deve acompanhar o train_loss. Se subir entre epochs = overfit
- **VRAM**: esperado ~22-32 GB dependendo do batch size e seq length

**Gotcha**: `trainer.evaluate()` pós-treino causa `CUDA IllegalMemoryAccess` nas camadas DeltaNet do Qwen3.5 (bug Unsloth em inferência). O val loss do treino (`eval_strategy="epoch"`) é confiável — não chame `evaluate()` separadamente.

In [ ]:
import torch

# Reset de estatísticas de memória para medir VRAM de pico
torch.cuda.reset_peak_memory_stats()

# Treinar
stats = trainer.train()

# Métricas finais
print(f"\n{'='*60}")
print(f"TREINO CONCLUÍDO")
print(f"{'='*60}")
print(f"Train loss final: {stats.metrics.get('train_loss', 'N/A'):.4f}")
print(f"Tempo total: {stats.metrics.get('train_runtime', 0)/60:.1f} min")

# Histórico de loss por step (overfit check)
print(f"\nHistórico de loss:")
for row in trainer.state.log_history:
    if 'loss' in row:
        print(f"  Step {row.get('step', '?'):>4}: train_loss = {row['loss']:.4f}")
    if 'eval_loss' in row:
        print(f"  Step {row.get('step', '?'):>4}: eval_loss  = {row['eval_loss']:.4f}  <-- val")

# VRAM de pico
vram = torch.cuda.max_memory_reserved() / 1024**3
print(f"\nVRAM pico: {vram:.1f} GB")
print(f"\nNOTA: NÃO chame trainer.evaluate() — causa CUDA IllegalMemoryAccess")
print(f"      nas camadas DeltaNet (bug Unsloth). Val loss acima é confiável.")

## 7. Visualizar curva de loss

A curva de loss é o principal indicador de qualidade do treino. Val loss subindo enquanto train loss desce = overfit.

In [ ]:
import matplotlib.pyplot as plt

# Extrair histórico do trainer
train_steps, train_losses = [], []
eval_steps, eval_losses = [], []

for row in trainer.state.log_history:
    if 'loss' in row and 'step' in row:
        train_steps.append(row['step'])
        train_losses.append(row['loss'])
    if 'eval_loss' in row and 'step' in row:
        eval_steps.append(row['step'])
        eval_losses.append(row['eval_loss'])

fig, ax = plt.subplots(figsize=(10, 5))

if train_steps:
    ax.plot(train_steps, train_losses, 'b-', alpha=0.7, label='Train loss', linewidth=1.5)
if eval_steps:
    ax.plot(eval_steps, eval_losses, 'ro-', markersize=8, label='Val loss', linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Curva de Loss — LoRA SFT', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Diagnóstico automático
if len(eval_losses) >= 2:
    if eval_losses[-1] > eval_losses[-2]:
        print("AVISO: Val loss subiu entre epochs — possível overfit.")
        print("Considere usar early stopping ou reduzir epochs.")
    else:
        print("OK: Val loss estável ou caindo — sem overfit detectado.")

## 8. Salvar adapter LoRA

O adapter LoRA é pequeno (~50-200 MB) e contém apenas as matrizes A e B treinadas. Pode ser carregado sobre qualquer cópia do modelo base.

In [ ]:
# Salvar adapter LoRA no Drive
ADAPTER_DIR = '/content/drive/MyDrive/ai-orchestrator-lora/adapter'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Verificar tamanho
import os
total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk(ADAPTER_DIR)
    for f in fn
)
print(f"Adapter salvo em: {ADAPTER_DIR}")
print(f"Tamanho total: {total_size / 1024**2:.1f} MB")

## 9. Resultados do AI-Orchestrator

O treino real do AI-Orchestrator produziu os seguintes resultados:

| Métrica | Valor |
|---------|-------|
| Dataset | 3.050 exemplos (2.745 train / 305 val) |
| Epochs | 2 (344 steps) |
| Tempo | 148 min (A100 40GB) |
| VRAM pico | 31.8 GB |
| Train loss (epoch 1) | 0.091 |
| Val loss (epoch 1) | 0.097 |
| Train loss (epoch 2) | 0.071 |
| Val loss (epoch 2) | 0.089 |

### Gotchas documentadas:
1. `trainer.evaluate()` pós-treino causa CUDA IllegalMemoryAccess (DeltaNet)
2. Arrow/`load_dataset('json')` falha em tool_calls heterogêneos
3. Checkpoints SEMPRE no Drive (2h17 perdidas por `output_dir` em `/content`)
4. Unsloth gera GGUF em `gguf_gguf/` (adiciona `_gguf` ao path)

No próximo capítulo, vamos avaliar o modelo treinado contra o baseline.